> ### **Building with Sarvam**
>
> **An open teaching kit for the Sarvam AI stack.**
>
> Notebook maintained by **Dr. Bhaveshkumar C. Dharmani** — Founder & AI Mentor, AIVidhya4Sarvam.
PhD (ICT), DA-IICT Gandhinagar · https://www.aividhya.in/ · https://www.aividhya4sarvam.in/ · bhavesh@aividhya.in · https://www.linkedin.com/in/bhaveshdharmani/
>
> Drafted with AI assistance and stress-tested cell by cell in live workshop sessions. Runs against your own Sarvam API key from dashboard.sarvam.ai, with a live ₹ cost meter after every call.
>
> Apache 2.0 · Issues and PRs welcome at github.com/dharmanibc/building-with-sarvam.

<div style="background:#12172E;color:#fff;padding:20px 24px;border-radius:8px">
<div style="color:#FF8A3D;font-size:12px;letter-spacing:2px;font-weight:700">LAB 02 · SPEECH IN</div>
<div style="font-size:26px;font-weight:700;margin-top:6px">Saaras — every mode, every path, every trap</div>
<div style="color:#FFB37A;font-size:14px;margin-top:8px">5 modes · REST / Batch / WebSocket · the 8 kHz cliff · diarization · pronunciation</div>
</div>

**Time:** 60 min &nbsp;·&nbsp; **Est. cost:** ≈ ₹8 &nbsp;·&nbsp; **Prereq:** Lab 00

## What you will have proved by the end

1. The five modes are **five different products**, not five formatting options
2. Telephony audio without `sample_rate` fails **silently** — HTTP 200, wrong text
3. Batch, REST and WebSocket suit completely different shapes of audio
4. Diarization costs 50% more and you usually do not need it

In [1]:
# %pip install audioop-lts
# %pip install --upgrade sarvamai

In [2]:
# ── Standard lab header. Run this first in every notebook. ─────────────────
import os, sys, json, time, math, wave, io
from pathlib import Path

# pip install sarvamai python-dotenv

from dotenv import load_dotenv, find_dotenv
# Finds your key without hardcoding anyone's filesystem. Tried in order:
#   1. SARVAM_API_KEY already set in the environment
#   2. the file named by SARVAM_ENV_FILE, if you set that variable
#   3. a .env beside this notebook, or in any parent folder
load_dotenv(os.environ.get("SARVAM_ENV_FILE") or find_dotenv(usecwd=True))

API_KEY = os.environ.get("SARVAM_API_KEY")
assert API_KEY, (
    "SARVAM_API_KEY not found.\\n"
    "Create a .env next to this notebook containing:  SARVAM_API_KEY=sk_...\\n"
    "or point SARVAM_ENV_FILE at an existing env file.\\n"
    "Free key + Rs 1000 credit: https://indus.sarvam.ai/"
)

from sarvamai import SarvamAI

client = SarvamAI(api_subscription_key=API_KEY)
DATA = Path("./data"); DATA.mkdir(exist_ok=True)
OUT  = Path("./out");  OUT.mkdir(exist_ok=True)
print("SDK ready ·", sys.version.split()[0])

SDK ready · 3.13.9


In [3]:
# ── The ₹ meter, imported ─────────────────────────────────────────────────
# Lab 00 writes cost_meter.py next to these notebooks. If this import fails,
# run Lab 00 once — it is the only lab that defines the meter.
try:
    from cost_meter import CostMeter, RATES, FREE_CREDIT
except ImportError:
    raise ImportError(
        "cost_meter.py not found.\n"
        "Run 00_Setup_and_the_Cost_Meter.ipynb once — its last section writes "
        "cost_meter.py into this folder, and every other lab imports it from there."
    )

cost = CostMeter()
print(f"cost meter armed · rates dated Aug 2026 · ₹{FREE_CREDIT:.0f} free credit")


cost meter armed · rates dated Aug 2026 · ₹1000 free credit


### 0 · Get some audio

You need three files. Use your own if you have them — the lab is far better with
real audio from your own domain.

| File | What it should be |
|---|---|
| `data/clean_16k.wav` | Clean 16 kHz speech, ~10s, any Indian language |
| `data/codemix.wav`   | Code-mixed speech: *"mera EMI due date kya hai"* |
| `data/call_8k.wav`   | Telephony audio, 8 kHz — or downsample the clean one below |

In [4]:
# Generate the sample files from TTS if you have no recordings
from sarvamai.play import save

SPECS = [
    ("clean_16k.wav", "नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ।", "hi-IN"),
    ("codemix.wav",   "मेरा EMI due date क्या है और late payment charge कितना लगेगा?", "hi-IN"),
]
for fname, text, lang in SPECS:
    a = client.text_to_speech.convert(text=text, language_code=lang,   #target_language_code=lang,
                                      model="bulbul:v3", speaker="pooja", speech_sample_rate=16000) #default sampling rate is: 22050 Hz
    save(a, str(DATA / fname)); cost.tts(len(text), v3=True)
    print("wrote", fname)

wrote clean_16k.wav
wrote codemix.wav


In [5]:
# Downsample to 8 kHz to simulate telephony (needs: apt install ffmpeg, or use audioop)
import audioop, wave

def downsample(src, dst, target=8000):
    with wave.open(str(src), "rb") as w:
        params = w.getparams(); frames = w.readframes(w.getnframes())
    conv, _ = audioop.ratecv(frames, params.sampwidth, params.nchannels,
                             params.framerate, target, None)
    with wave.open(str(dst), "wb") as o:
        o.setnchannels(params.nchannels); o.setsampwidth(params.sampwidth)
        o.setframerate(target); o.writeframes(conv)
    return dst

downsample(DATA / "clean_16k.wav", DATA / "call_8k.wav")
print("wrote call_8k.wav @ 8000 Hz")

wrote call_8k.wav @ 8000 Hz


In [6]:
# Helper: how long is a wav, in seconds? (we bill per second)
def duration(path):
    with wave.open(str(path), "rb") as w:
        return w.getnframes() / w.getframerate()

for f in ["clean_16k.wav", "codemix.wav", "call_8k.wav"]:
    print(f"{f:<16} {duration(DATA/f):.2f}s")

clean_16k.wav    3.67s
codemix.wav      4.78s
call_8k.wav      3.67s


---
## 1 · The five modes

Same audio. Five different jobs. This is the heart of Saaras and most people
discover it three weeks into a project.

In [7]:
def transcribe(path, mode="transcribe", **kw):
    with open(path, "rb") as f:
        r = client.speech_to_text.transcribe(
            file=f, 
            model="saaras:v3", 
            mode=mode, 
            **kw
        )
    cost.stt(duration(path), diarized=kw.get("diarization", False))
    return r

MODES = ["transcribe", "translate", "verbatim", "translit", "codemix"]
table = {}
for m in MODES:
    table[m] = transcribe(DATA / "codemix.wav", mode=m, language_code="hi-IN").transcript

for m, t in table.items():
    print(f"{m:>11} │ {t}")

 transcribe │ मेरा ईएमआई ड्यू डेट क्या है और लेट पेमेंट चार्ज कितना लगेगा?
  translate │ What is my EMI due date and how much will the late payment charge be?
   verbatim │ मेरा ई एम आई ड्यू डेट क्या है और लेट पेमेंट चार्ज कितना लगेगा
   translit │ Mera EMI due date kya hai aur late payment charge kitna lagega
    codemix │ मेरा EMI due date क्या है और late payment charge कितना लगेगा?


| mode | Returns | Reach for it when |
|---|---|---|
| `transcribe` | Native script, lightly normalised | **Default.** Chat logs, search, storage |
| `translate` | English | Analytics, dashboards, English-only downstream |
| `verbatim` | Every filler, stutter, repetition | Compliance, QA scoring, legal record |
| `translit` | Roman script | Your DB or UI can't render Devanagari |
| `codemix` | The actual mix as spoken | Training data, authentic UX |

**Question for the room:** which mode for a recorded compliance call? Most people say
`transcribe`. The answer is `verbatim` — a regulator wants the disfluencies.

---
## 2 · The 8 kHz question — measure it, do not take anyone's word for it

Real Indian call traffic is 8 kHz. Every Western speech stack degrades on it, and
the internet is full of advice telling you to declare a `sample_rate` so the model
knows what it is looking at.

**On the Saaras REST path, that parameter does not exist.** Try it and you get:

```
TypeError: SpeechToTextClient.transcribe() got an unexpected keyword argument 'sample_rate'
```

That is not an oversight. A `.wav` file is a *container* — its 44-byte RIFF header
already declares the sample rate, channel count and bit depth. Saaras reads the
header. There is nothing for you to declare.

So the honest experiment is not *"what breaks without the flag"* — it is
**"how much accuracy does telephony bandwidth actually cost on this model?"**
Let us measure it.

In [8]:
right_16k = client.speech_to_text.transcribe(
            file= open(DATA / "clean_16k.wav", "rb"), 
            language_code="hi-IN", 
            model="saaras:v3", 
            mode="transcribe",
        )
print("For Sampling Rate of 16k: ", right_16k.transcript)

For Sampling Rate of 16k:  नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ।


In [9]:
right_8k = client.speech_to_text.transcribe(
            file= open(DATA / "call_8k.wav", "rb"), 
            language_code="hi-IN", 
            model="saaras:v3", 
            mode="transcribe",
            # sample_rate=8000,
        )
print("For Sampling Rate of 8k: ", right_8k.transcript)

For Sampling Rate of 8k:  नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ।


In [10]:
# Quantify the damage with a crude word error rate
def wer(ref, hyp):
    r, h = ref.split(), hyp.split()
    d = [[0]*(len(h)+1) for _ in range(len(r)+1)]
    for i in range(len(r)+1): d[i][0] = i
    for j in range(len(h)+1): d[0][j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1] + (r[i-1] != h[j-1]))
    return d[-1][-1] / max(len(r), 1)

score = wer(right_16k.transcript, right_8k.transcript)
print(f"WER  8 kHz vs 16 kHz : {score:.2%}")
print()
if score == 0:
    print("Identical. On this clip, halving the bandwidth cost nothing at all.")
elif score < 0.10:
    print("Under 10% — telephony bandwidth is essentially free on this clip.")
else:
    print("Above 10% — worth re-testing on real call recordings before you quote a number.")
print("\nThis is a synthetic downsample of clean TTS audio. Real telephony also brings")
print("codec loss, packet jitter and background noise. Measure on YOUR recordings")
print("before you put an accuracy number in front of a customer.")


WER  8 kHz vs 16 kHz : 0.00%

Identical. On this clip, halving the bandwidth cost nothing at all.

This is a synthetic downsample of clean TTS audio. Real telephony also brings
codec loss, packet jitter and background noise. Measure on YOUR recordings
before you put an accuracy number in front of a customer.


### So where *does* `sample_rate` matter?

On the **streaming** paths — and only there. When you stream, you are not sending a
container with a header; you are sending a bare sequence of samples. Nothing in that
byte stream says how fast to play it back, so you must declare it:

| Path | Sample rate | Why |
|---|---|---|
| `speech_to_text.transcribe(...)` — REST | **Not a parameter** | The `.wav` header already declares it |
| `speech_to_text_streaming.connect(sample_rate=...)` | **Required** | Raw sample stream, no container |
| `...socket.transcribe(audio, sample_rate=16000)` | **Per chunk** | Defaults to 16000 — override for telephony |
| `speech_to_text_realtime_streaming.connect(sample_rate=...)` | **Required** | Same reason |

> **The rule worth remembering.** Container in, rate declared for you. Raw samples in,
> you declare the rate. Get this wrong on a streaming socket and you *will* get the
> silent-garbage failure the internet warned you about — it is just in section 5,
> not here.

---
## 3 · Language handling — declare, or auto-detect?

In [11]:
# Explicit language code
a = transcribe(DATA / "codemix.wav", language_code="hi-IN")

# Auto-detect — omit language_code entirely
b = transcribe(DATA / "codemix.wav")

print("explicit  :", a.transcript)
print("auto      :", b.transcript)
print("detected  :", getattr(b, "language_code", "—"))

explicit  : मेरा ईएमआई ड्यू डेट क्या है और लेट पेमेंट चार्ज कितना लगेगा?
auto      : मेरा ईएमआई ड्यू डेट क्या है और लेट पेमेंट चार्ज कितना लगेगा?
detected  : hi-IN


**Rule of thumb.** Auto-detect for unknown inbound audio. Declare explicitly when you
already know (an IVR where the caller picked Hindi) — it is more accurate and lets
you skip a detection round-trip.

**23 languages:** `hi-IN bn-IN ta-IN te-IN mr-IN gu-IN kn-IN ml-IN od-IN pa-IN as-IN
ur-IN ne-IN kok-IN ks-IN sd-IN sa-IN sat-IN mni-IN brx-IN mai-IN doi-IN en-IN`

---
## 4 · Delivery path 2 — Batch, for long audio

Up to 20 files × 60 minutes. Async: submit → poll → fetch. This is also the only
path with **speaker diarization**.

In [12]:
# # Poll with backoff until terminal
# import time
# delay = 2
# while True:
#     st = job.get_status()
#     print(f"  status={st.job_state}  waited={delay}s")
#     if st.job_state in ("Completed", "Failed"):
#         break
#     time.sleep(delay); 
#     delay = min(delay * 1.5, 20)

# if st.job_state == "Completed":
#     job.download_outputs(output_dir=str(OUT / "batch"))
#     cost.stt(duration(DATA / "codemix.wav"), diarized=True)
#     print("outputs in", OUT / "batch")


In [13]:
# # Alternative to the above way of polling for the status change, the new
# approach # Wait for completion
# job.wait_until_complete()

# # Check file-level results
# file_results = job.get_file_results()


# print(f"\nSuccessful: {len(file_results['successful'])}")
# for f in file_results['successful']:
#     print(f"  ✓ {f['file_name']}")
#     job.download_outputs(output_dir=str(OUT / "batch"))
#     # cost.stt( sum(duration(file_paths)), diarized=True) #cost.stt( duration(DATA / "codemix.wav"), diarized=True)
#     cost.stt(duration(DATA / f['file_name']), diarized=True)
#     print(f"\nDownloaded {len(file_results['successful'])} file(s) to: ./out/batch ")

# print(f"\nFailed: {len(file_results['failed'])}")
# for f in file_results['failed']:
#     print(f"  ✗ {f['file_name']}: {f['error_message']}")    

### Polling: use the SDK's own waiter, not a hand-rolled loop

Older cookbook snippets hand-roll a `while True` with `get_status()` and a backoff.
It works, but the SDK ships `wait_until_complete()` which does exactly that — with a
proper timeout and a `TimeoutError` you can actually catch:

```python
job.wait_until_complete(poll_interval=5, timeout=600)   # ← use this
```

Companion helpers you get for free: `job.is_complete()`, `job.is_successful()`,
`job.is_failed()`, `job.get_file_results()`, `job.get_output_mappings()`.
Prefer these over parsing `job_state` strings yourself — the string casing has
changed between API versions, and these methods normalise it for you.

In [14]:
# Batch job with diarization. Note the price change: ₹30/hr → ₹45/hr
job = client.speech_to_text_job.create_job(
    model="saaras:v3",
    mode="transcribe",          # "transcribe", "translate", "verbatim", "translit", "codemix"
    language_code="hi-IN",
    with_diarization=True,
    num_speakers=2,
)

BATCH_FILES = ["clean_16k.wav", "codemix.wav", "query_16k.wav"]
# file_paths = [str(DATA / f) for f in BATCH_FILES]   #
file_paths = [str(DATA / "codemix.wav"), str(DATA / "clean_16k.wav"), str(DATA / "call_8k.wav")]
job.upload_files(file_paths)
job.start()
print("job started:", job.job_id)
print("files queued:", len(file_paths))

job started: 20260828_a5db1ddd-10e6-440c-a0b9-6a14ece8203e
files queued: 3


In [15]:
# ── Wait, then read PER-FILE results ─────────────────────────────────────
try:
    final = job.wait_until_complete(poll_interval=5, timeout=600)
    print("job_state:", final.job_state)
except TimeoutError as e:
    print("timed out:", e)

results = job.get_file_results()          # {"successful": [...], "failed": [...]}

print(f"\n{'file':<20}{'status':<12}{'output':<28}")
print("─" * 60)
for r in results["successful"]:
    print(f"{r['file_name']:<20}{r['status']:<12}{str(r['output_file']):<28}")
for r in results["failed"]:
    print(f"{r['file_name']:<20}{r['status']:<12}{r['error_message']}")

print(f"\n{len(results['successful'])} succeeded · {len(results['failed'])} failed")

job_state: Completed

file                status      output                      
────────────────────────────────────────────────────────────
call_8k.wav         Success     0.json                      
clean_16k.wav       Success     1.json                      
codemix.wav         Success     2.json                      

3 succeeded · 0 failed


> **Why `get_file_results()` matters more than it looks.** In a 20-file job, two files
> can fail — a corrupt upload, an unsupported codec — while the job as a whole still
> reports `Completed`. If you only check the job state, you silently lose those two
> files. This is the batch-mode cousin of the `partially_completed` trap you will meet
> again in Lab 06 with Document AI. **Always reconcile counts: files in == files out.**

In [16]:
# ── Download every successful output, then read the diarized turns ───────
if results["successful"]:
    job.download_outputs(output_dir=str(OUT / "batch"))
    # Bill for what actually PROCESSED, not what we intended to upload.
    # A file that failed server-side should not appear on your cost report.
    billed = 0.0
    for r in results["successful"]:
        p = DATA / r["file_name"]
        if p.exists():
            billed += duration(p)
    cost.stt(billed, diarized=True)
    
    # billed = sum([duration(DATA / f) for f in BATCH_FILES]) #sum(duration(file_paths))  #sum(duration(DATA / f) for f in BATCH_FILES)
    # cost.stt(billed, diarized=True)
    print(f"downloaded {len(results['successful'])} files to {OUT / 'batch'}")

    # get_output_mappings() tells you which output belongs to which input
    for m in job.get_output_mappings():
        print(f"  {m['input_file']}  →  {m['output_file']}")
else:
    print("No files succeeded — nothing to download, nothing billed.")
    for r in results["failed"]:
        print(f"  {r['file_name']}: {r['error_message']}")

downloaded 3 files to out/batch
  call_8k.wav  →  0.json
  clean_16k.wav  →  1.json
  codemix.wav  →  2.json


In [17]:
# Read the diarized output — who said what, when
import glob
for p in glob.glob(str(OUT / "batch" / "*.json")):
    print(p)
    data = json.load(open(p))
    entries = data.get("diarized_transcript", {}).get("entries", [])
    if not entries:
        print("   (no diarized entries — check the raw JSON keys:", list(data)[:6], ")")
    for seg in data.get("diarized_transcript", {}).get("entries", [])[:10]:
        print(f"[{seg.get('start_time_seconds', 0):>6.2f}s] "
              f"spk{seg.get('speaker_id', '?')}: {seg.get('transcript', '')} ")

out/batch/clean_16k.wav.json
[  0.01s] spk0: नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ। 
out/batch/call_8k.wav.json
[  0.01s] spk0: नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ। 
out/batch/codemix.wav.json
[  0.01s] spk0: मेरा ईएमआई ड्यू डेट क्या है और लेट पेमेंट चार्ज कितना लगेगा? 


> **Cost discipline.** Diarization is ₹45/hr vs ₹30/hr — a 50% premium. Only pay it
> when you genuinely need *who* said something. For a single-speaker voice note it is
> pure waste, and at a million minutes a month that waste is real money.

---
## 5 · Delivery path 3 — WebSocket streaming

The one your voice agent needs. Partial results come back while the person is still
talking.

**Three things bite everyone here, and all three are worth causing on purpose:**

1. **`SarvamAI` is synchronous.** `connect()` is a *sync* context manager. Write
   `async with` / `await` against it and you get
   `TypeError: 'generator' object does not support the asynchronous context manager
   protocol`. For `async`, use `AsyncSarvamAI` instead — a different class.
2. **Audio must be base64 text, not raw bytes.** The socket carries JSON, and raw
   PCM bytes are not JSON-serialisable.
3. **This is where `sample_rate` finally matters** — you are sending bare samples
   with no header, exactly as section 2 promised.

In [18]:
# ── The failure, on purpose. Read the error, then look at the fix below. ─
import asyncio
try:
    async def broken():
        async with client.speech_to_text_streaming.connect(      # ← sync client!
                model="saaras:v3", language_code="hi-IN") as ws:
            await ws.transcribe(audio=b"\x00\x00")               # ← raw bytes!
    await broken()
except TypeError as e:
    print("TypeError (expected):", e)
    print("\n↑ `SarvamAI` is sync. Either use `with`, or switch to `AsyncSarvamAI`.")
except Exception as e:
    print(f"{type(e).__name__}: {e}")

TypeError (expected): '_GeneratorContextManager' object does not support the asynchronous context manager protocol

↑ `SarvamAI` is sync. Either use `with`, or switch to `AsyncSarvamAI`.


In [19]:
# ── The SYNC way — simplest, and correct for a notebook ──────────────────
# Two things confirmed by reading the installed SDK source directly
# (sarvamai==0.1.28, speech_to_text_streaming/socket_client.py + types/):
#
# 1. Response `type` is only ever "data" | "error" | "events" — never
#    "end_of_stream"/"close". Waiting for that killed the first version.
# 2. The server does NOT reply once per chunk sent — it buffers internally
#    and emits partials on its own cadence. Calling `ws.recv()` right after
#    every single `ws.transcribe()` assumes a 1:1 pairing that doesn't hold.
#
# `Connection.recv(timeout=N)` is the real, documented way to poll without
# hanging: it raises TimeoutError if nothing arrives, ConnectionClosed if the
# socket closes. Send everything first, then drain with that.
#
# A fixed recv_timeout is itself a trap: on clean_16k.wav (~3.4s) 3s was
# plenty, but on a 10x-looped ~34s clip it produced 0 messages — not because
# anything broke, but because the server legitimately needed more than 3s to
# finish transcribing 10x more audio before it sent its (single, flush-time)
# reply. Scale the wait with the audio's own duration instead of guessing a
# constant that only holds for whatever clip you tested last.
import base64, wave, json
from sarvamai.core.pydantic_utilities import parse_obj_as
from sarvamai.speech_to_text_streaming.socket_client import (
    SpeechToTextStreamingSocketClientResponse,
)

def stream_transcribe(path, language="hi-IN", model="saaras:v3", recv_timeout=None):
    """Chunked WebSocket transcription: send everything, then drain replies."""
    with wave.open(str(path), "rb") as w:
        rate     = w.getframerate()          # read it from the header ONCE
        n_frames = w.getnframes()
        chunk    = int(rate * 0.2)           # 200 ms per chunk
        pcm      = [w.readframes(chunk) for _ in range(0, n_frames, chunk)]

    audio_s = n_frames / rate
    if recv_timeout is None:
        # Baseline 5s (network + model warm-up) + 1s of wait per second of
        # audio, capped at 60s so a bug elsewhere can't hang the cell forever.
        recv_timeout = min(60.0, 5.0 + audio_s)
    print(f"  audio duration {audio_s:.1f}s → recv_timeout {recv_timeout:.1f}s")

    transcripts = []
    with client.speech_to_text_streaming.connect(
        model         = model,
        language_code = language,
        sample_rate   = str(rate),           # ← REQUIRED here. Note: a STRING.
        input_audio_codec = "pcm_s16le",
    ) as ws:
        n_sent = 0
        for frames in pcm:
            if not frames:
                continue
            ws.transcribe(
                audio       = base64.b64encode(frames).decode(),   # ← base64 TEXT
                encoding    = "audio/wav",
                sample_rate = rate,
            )
            n_sent += 1
        ws.flush()                            # force-finalize the last open segment
        print(f"  sent {n_sent} chunk(s) + flush, now draining replies...")

        n_msgs = 0
        while True:
            try:
                raw = ws._websocket.recv(timeout=recv_timeout)   # real per-call timeout
            except TimeoutError:
                print(f"  (no message within {recv_timeout:.1f}s — treating as done)")
                break
            except Exception as e:
                # DO NOT silently swallow — show exactly what stopped the drain,
                # so a real bug doesn't look identical to a clean finish.
                print(f"  drain stopped: {type(e).__name__}: {e}")
                break
            n_msgs += 1
            raw = json.loads(raw) if isinstance(raw, str) else raw
            msg = parse_obj_as(SpeechToTextStreamingSocketClientResponse, raw)
            if msg.type == "data" and getattr(msg.data, "transcript", None):
                transcripts.append(msg.data.transcript)
                print("  →", msg.data.transcript)
            elif msg.type == "error":
                print("  server error:", msg.data)
            else:
                print("  (other message):", msg.type, msg.data)
        print(f"  received {n_msgs} message(s) total")
    return " ".join(transcripts)

try:
    text = stream_transcribe(DATA / "clean_16k.wav")
    cost.stt(duration(DATA / "clean_16k.wav"))
    print("\nFINAL:", text)
except Exception as e:
    print(f"{type(e).__name__}: {e}")
    print("Streaming APIs move fast between SDK versions — check docs.sarvam.ai/api-reference.")

  audio duration 3.7s → recv_timeout 8.7s
  sent 19 chunk(s) + flush, now draining replies...
  → नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ।
  (no message within 8.7s — treating as done)
  received 1 message(s) total

FINAL: नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ।


In [20]:
# ── Make a longer clip by looping clean_16k.wav N times ───────────────────
# Real streaming sessions carry many seconds of audio with several VAD turns.
# A single ~5s sentence never gives the server a chance to emit intermediate
# events, so hangs there don't tell us what a real session looks like.
# Concatenating raw PCM frames N times gives a longer, still-valid WAV with
# the same header (rate/channels/sampwidth unchanged) — no re-encoding needed.

def loop_audio(src, dst, times=10):
    with wave.open(str(src), "rb") as w:
        params = w.getparams()
        frames = w.readframes(w.getnframes())
    with wave.open(str(dst), "wb") as o:
        o.setparams(params)
        o.writeframes(frames * times)
    return dst

loop_audio(DATA / "clean_16k.wav", DATA / "clean_16k_long.wav", times=10)
print(f"clean_16k_long.wav : {duration(DATA / 'clean_16k_long.wav'):.2f}s "
      f"(clean_16k.wav was {duration(DATA / 'clean_16k.wav'):.2f}s)")

try:
    text = stream_transcribe(DATA / "clean_16k_long.wav")
    cost.stt(duration(DATA / "clean_16k_long.wav"))
    print("\nFINAL:", text)
except Exception as e:
    print(f"{type(e).__name__}: {e}")
    print("Streaming APIs move fast between SDK versions — check docs.sarvam.ai/api-reference.")

clean_16k_long.wav : 36.69s (clean_16k.wav was 3.67s)
  audio duration 36.7s → recv_timeout 41.7s
  sent 184 chunk(s) + flush, now draining replies...
  → नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ। नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ। नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ। नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ। नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ। नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ। नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ। नमस्ते मैं अपने लोन के बारे में जानकारी चाहता हूँ, नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ।
  (no message within 41.7s — treating as done)
  received 1 message(s) total

FINAL: नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ। नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ। नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ। नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ। नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ। नमस्ते, मैं अपने लोन के बारे में 

### 5.1 · The realtime endpoint — `saaras:v3-realtime`

There are **two** streaming surfaces and they are not the same product:

| | `speech_to_text_streaming` | `speech_to_text_realtime_streaming` |
|---|---|---|
| Model | `saaras:v3` / `saaras:v4` | `saaras:v3-realtime` |
| Built for | Streaming a file or a long feed | **Live conversation** |
| Events | `data` messages | `transcript.partial`, `transcript.final`, `vad.speech_start/end` |
| Endpointing | `flush()` | Server-side VAD, or `manual` |
| Encoding | `pcm_s16le`, `wav` | `linear16`, `mulaw`, `alaw` ← **telephony codecs** |
| Use it for | Batch-ish streaming, captioning | The voice agent in Lab 08 |

The realtime endpoint is the one that speaks `mulaw` — i.e. the one that plugs
straight into an Indian phone bridge. It is also the model behind the realtime
captioning example in Sarvam's own cookbook.

In [21]:
# ── Realtime streaming — VAD-driven, partial + final transcripts ─────────
from sarvamai.types.realtime_audio_input import RealtimeAudioInput
from sarvamai.types.realtime_end import RealtimeEnd
import threading

def realtime_transcribe(path, language="hi-IN", end_timeout=8.0):
    with wave.open(str(path), "rb") as w:
        rate     = w.getframerate()
        n_frames = w.getnframes()
        chunk    = int(rate * 0.1)                       # 100 ms — realtime cadence
        pcm      = [w.readframes(chunk) for _ in range(0, n_frames, chunk)]

    partials, finals = [], []
    with client.speech_to_text_realtime_streaming.connect(
        model         = "saaras:v3-realtime",
        language_code = language,
        encoding      = "linear16",       # "mulaw" for an 8 kHz phone bridge
        sample_rate   = str(rate),
        endpointing   = "vad",            # let the server decide turn boundaries
        stream_type   = "balanced",       # "fast" | "balanced" | "simulated"
    ) as ws:
        for frames in pcm:
            if not frames:
                continue
            ws.send_realtime_audio_input(
                RealtimeAudioInput(audio=base64.b64encode(frames).decode()))
        ws.send_realtime_end(RealtimeEnd())              # signal end of input

        # `for msg in ws` blocks on the socket's recv(). If the server never
        # emits "session.end" after RealtimeEnd (common on a short, single
        # utterance with no clean trailing silence for the VAD to key off),
        # this iterator blocks FOREVER waiting on a message that never comes
        # — a timeout check inside the loop body can't catch that, since the
        # body only runs again once a *new* message arrives. So drain on a
        # background thread and force-close the socket from the main thread
        # if it doesn't finish in time.
        def _drain():
            for msg in ws:
                ev = getattr(msg, "event", "")
                if ev == "transcript.partial":
                    partials.append(msg.text)
                    print(f"  … {msg.text}")
                elif ev == "transcript.final":
                    finals.append(msg.text)
                    ts = f"[{msg.start_s:.2f}s\u2013{msg.end_s:.2f}s] " if msg.start_s is not None and msg.end_s is not None else ""
                    print(f"  \u2713 {ts}{msg.text}")
                elif ev in ("vad.speech_start", "vad.speech_end"):
                    print(f"  · {ev}")
                elif ev in ("session.end", "error"):
                    if ev == "error":
                        print("  server error:", getattr(msg, "message", msg))
                    break

        th = threading.Thread(target=_drain, daemon=True)
        th.start()
        th.join(timeout=end_timeout)
        if th.is_alive():
            print(f"  (no session.end after {end_timeout:.0f}s — forcing close)")
            try:
                ws.close()             # unblocks the thread's recv()
            except Exception:
                pass
            th.join(timeout=2)
    return partials, finals

try:
    partials, finals = realtime_transcribe(DATA / "clean_16k.wav")
    cost.stt(duration(DATA / "clean_16k.wav"))
    print(f"\n{len(partials)} partials → {len(finals)} final utterance(s)")
    print("FINAL:", " ".join(finals))
except Exception as e:
    print(f"{type(e).__name__}: {e}")
    print("If this endpoint is not enabled on your plan, skip to section 6 —")
    print("Lab 08 rebuilds the same loop inside a full voice agent.")
    

  · vad.speech_start
  · vad.speech_end
  … 
  … नमस्ते मैं
  … नमस्ते मैं अपने 
  … नमस्ते मैं अपने लोन 
  … नमस्ते मैं 
  … नमस्ते मैं अपने लोन
  … नमस्ते मैं अपने लोन के 
  … नमस्ते मैं अपने लोन के बारे में
  … नमस्ते मैं अपने लोन के बारे में जानकारी 
  … नमस्ते मैं अपने लोन के बारे में जानकारी चाहता हूं
  … नमस्ते मैं अपने लोन के बारे में जानकारी चाहता हूं
  … नमस्ते मैं अपने लोन के बारे
  … नमस्ते मैं अपने लोन के बारे में 
  ✓ नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ।

13 partials → 1 final utterance(s)
FINAL: नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ।


**Partials vs finals — the distinction your UI depends on.** Partials are the model's
running best guess; they get *revised* as more audio arrives. Finals are committed and
never change. Render partials in grey and finals in black, and your captioning UI
suddenly feels like every professional one you have used.

**VAD is what decides where a turn ends.** `endpointing="vad"` lets the server call it
from silence duration; `endpointing="manual"` puts you in charge. Tuning
`silence_duration_ms` is the difference between an agent that interrupts people and one
that feels patient — you will tune exactly this in Lab 08.

---
## 6 · Pronunciation dictionaries

Brand names, scheme names, SKUs and place names get mangled by default. A dictionary
fixes them for both recognition and synthesis.

In [22]:
import json, tempfile

# 1. Build the pronunciation dictionary as JSON: language_code -> {word: pronunciation}
pron_data = {
    "pronunciations": {
        "hi-IN": {
            "IRDAI":    "आई आर डी ए आई",
            "PM-KISAN": "पी एम किसान",
            "AIVidhya4Sarvam": "ए आई विद्या फॉर सर्वम्",
        }
    }
}

with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    json.dump(pron_data, f, ensure_ascii=False)
    dict_path = f.name

# 2. Upload it — only bulbul:v3 honours dictionaries
with open(dict_path, "rb") as f:
    d = client.pronunciation_dictionary.create(file=f)

print("dictionary_id:", d.dictionary_id)

# 3. Same sentence, with vs without the dictionary

# text = "IRDAI के नियमों के तहत PM-KISAN योजना का लाभ मिलेगा।"
# text = "AIVidhya4Sarvam का लक्ष्य ज्ञान और रचनात्मकता का लोकतंत्रीकरण करना है।" #"ए.आई. विद्या फॉर सर्वम् का लक्ष्य ज्ञान और रचनात्मकता का लोकतंत्रीकरण करना है।"
text = "AIVidhya4Sarvam ka lakshya gyaan aur rachanatmakta ka loktantrikaran karna hai."

without_dict = client.text_to_speech.convert(
    text=text, language_code="hi-IN", model="bulbul:v3", speaker="shubh",
)
with_dict = client.text_to_speech.convert(
    text=text, language_code="hi-IN", model="bulbul:v3", speaker="shubh",
    dict_id=d.dictionary_id,
)

from sarvamai.play import save
save(without_dict, str(OUT / "no_dict.wav"))
save(with_dict, str(OUT / "with_dict.wav"))
print("Listen to out/no_dict.wav vs out/with_dict.wav — IRDAI/PM-KISAN should sound native in the second.")


dictionary_id: p_d15e6ab8
Listen to out/no_dict.wav vs out/with_dict.wav — IRDAI/PM-KISAN should sound native in the second.


In [23]:
# Play it right here in the notebook
from IPython.display import Audio, display
display(Audio(str(OUT / "no_dict.wav")))

In [24]:
# Play it right here in the notebook
from IPython.display import Audio, display
display(Audio(str(OUT / "with_dict.wav")))

---
## 5 · The bill

In [25]:
cost.report()

TTS          ₹   0.1530  51 chars
TTS          ₹   0.1830  61 chars
STT          ₹   0.0398  4.8s
STT          ₹   0.0398  4.8s
STT          ₹   0.0398  4.8s
STT          ₹   0.0398  4.8s
STT          ₹   0.0398  4.8s
STT          ₹   0.0398  4.8s
STT          ₹   0.0398  4.8s
STT          ₹   0.1515  12.1s
STT          ₹   0.0306  3.7s
STT          ₹   0.3058  36.7s
STT          ₹   0.0306  3.7s
TOTAL        ₹   1.1332
              (₹1000 free credit → ₹998.87 left)
              Estimated from published rates; actual billing usually lower


1.1331552083333334

---
## ✅ Checkpoint

- [ ] Five modes produced five genuinely different transcripts
- [ ] You measured the real WER cost of 8 kHz telephony audio — and can state the number
- [ ] You can explain **why** REST needs no `sample_rate` but streaming does
- [ ] A **three-file** batch job completed via `wait_until_complete()`, and you read the
      per-file table from `get_file_results()`
- [ ] Streaming produced partial results before the audio finished
- [ ] You saw partials get *revised* into finals on the realtime endpoint

## 🧪 Try this

1. Record 30 seconds of yourself, in your own dialect. Run all five modes. Where does it fail?
2. **Re-run the 8 kHz comparison on a real call recording**, not a synthetic downsample.
   Codec loss and background noise are what actually move WER — does your number hold?
3. Push the batch job to 10 files. Does throughput scale linearly, or does the
   10-requests-per-minute ceiling bite first?
4. Deliberately corrupt one file in the batch (truncate it) and confirm
   `get_file_results()["failed"]` catches it while the job still reports `Completed`.
5. Switch the realtime call to `encoding="mulaw"` with 8 kHz audio — the exact
   configuration a phone bridge sends. Does the transcript hold up?
6. Build a 10-file eval set from real audio and compute mean WER. **This is the artefact
   that wins enterprise pilots** — a documented accuracy number on *their* data.